# Pilot v1 Analysis — 5-condition multi-select

Conditions: `birds` (control), `birds_easier`, `birds_listed`, `birds_distractors`, `birds_repeat`  
Scoring: all-or-nothing per question (primary) + per-statement Hamming (secondary)

In [ ]:
import csv
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

HERE      = Path('.')  # run from pilot_v1/
CSV_PATH  = HERE / 'QA+Pilot+v2_June+15,+2026_12.17.csv'
YAML_PATH = HERE.parent / 'birds_q_multiselect.yaml'

# Q01..Q10 columns map 1-to-1 to QS01..QS10 in the YAML
QCOLS      = [f'Q{n:02d}' for n in range(1, 11)]
QIDS       = [f'QS{n:02d}' for n in range(1, 11)]
COL_TO_QID = dict(zip(QCOLS, QIDS))
LETTERS    = list('ABCDE')

CONDITIONS = ['birds', 'birds_easier', 'birds_listed', 'birds_distractors', 'birds_repeat']
COND_COLORS = {
    'birds':             '#1f77b4',
    'birds_easier':      '#2ca02c',
    'birds_listed':      '#ff7f0e',
    'birds_distractors': '#d62728',
    'birds_repeat':      '#9467bd',
}
COND_LABELS = {c: c.replace('birds_', '') for c in CONDITIONS}
COND_LABELS['birds'] = 'control'

# Attention check correct answers (used to exclude failed respondents)
AT1_CORRECT = 'Green versus rose'
AT2_CORRECT = 'Plumage strategy'

In [ ]:
# Load YAML and CSV
spec   = yaml.safe_load(YAML_PATH.read_text())
by_qid = {q['q_id']: q for q in spec['questions']}
PRIMARY = spec['metadata']['primary_subset']   # QS01-QS10 minus QS07

with CSV_PATH.open(newline='', encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))

# Qualtrics exports 3 header rows; data starts at row index 2.
# Exclude: no Prolific ID (test/preview runs), failed either attention check.
all_finished = [r for r in rows[2:] if r.get('Finished', '').lower() in {'true', '1'}]
data_rows = [
    r for r in all_finished
    if r.get('PROLIFIC_PID', '').strip()
    and r.get('AT1', '').strip() == AT1_CORRECT
    and r.get('AT2', '').strip() == AT2_CORRECT
]

print(f'Finished: {len(all_finished)}, kept after exclusions: {len(data_rows)}')
print('By condition:', Counter(r['assigned_doc'] for r in data_rows))

In [ ]:
def parse_selected(response_text, options_dict):
    """
    Qualtrics stores multi-select answers as the option texts joined by ','.  
    Map back to option letters by substring matching (strip trailing period and
    markdown underscores so distractors-condition longer texts still match).
    """
    if not response_text or not response_text.strip():
        return set()
    resp = response_text.strip()
    selected = set()
    for letter, opt_text in options_dict.items():
        key = re.sub(r'_', '', opt_text.strip()).rstrip('.')
        if key in resp:
            selected.add(letter)
    return selected


def all_or_nothing(selected, answer_key):
    return int(set(selected) == set(answer_key))


def hamming_acc(selected, answer_key, n_opts=5):
    """Fraction of options (A-E) correctly classified (selected iff in answer key)."""
    key_set = set(answer_key)
    sel_set = set(selected)
    return sum(1 for L in LETTERS[:n_opts] if (L in sel_set) == (L in key_set)) / n_opts


# Build per-respondent records
respondent_data = []
for r in data_rows:
    rec = {
        'cond': r['assigned_doc'],
        'pid':  (r.get('PROLIFIC_PID') or r.get('ResponseId', ''))[:12],
    }
    for qcol, qid in COL_TO_QID.items():
        q   = by_qid[qid]
        sel = parse_selected(r.get(qcol, ''), q['options'])
        rec[qid] = {
            'selected': sel,
            'aon':      all_or_nothing(sel, q['answer']),
            'hamming':  hamming_acc(sel, q['answer']),
        }
    rec['total_aon']    = sum(rec[qid]['aon']    for qid in PRIMARY)
    rec['total_hamming'] = np.mean([rec[qid]['hamming'] for qid in PRIMARY])
    respondent_data.append(rec)

print(f'Parsed {len(respondent_data)} respondents')

In [ ]:
# ── Answer distributions: one consolidated PNG ────────────────────────────────
# Layout: 10 rows (questions) × 5 cols (conditions).
# Bars colored by condition (same scheme as score plot); correct options marked ✓.

def wrap(s, n=18):
    s = re.sub(r'_', '', s.strip())
    words, lines, line = s.split(), [], ''
    for w in words:
        if len(line) + len(w) + 1 > n: lines.append(line); line = w
        else: line = (line + ' ' + w).strip()
    if line: lines.append(line)
    return '\n'.join(lines)

fig, axes = plt.subplots(len(QIDS), len(CONDITIONS), figsize=(22, 40), sharey='row')

for row, (qcol, qid) in enumerate(COL_TO_QID.items()):
    q = by_qid[qid]
    opts = q['options']
    answer_set = set(q['answer'])
    letters = LETTERS[:len(opts)]
    x = np.arange(len(letters))
    tag = f"e={q['metadata']['tags'].get('edges','?')} cm={q['metadata'].get('cue_match','?')}"

    for col, cond in enumerate(CONDITIONS):
        ax = axes[row, col]
        cond_recs = [rec for rec in respondent_data if rec['cond'] == cond]
        n = len(cond_recs)
        color = COND_COLORS[cond]

        rates = [sum(1 for rec in cond_recs if L in rec[qid]['selected']) / n * 100 if n else 0 for L in letters]
        # Correct options: full color; incorrect: 40% alpha (lighter)
        bar_colors = [color if L in answer_set else (*plt.matplotlib.colors.to_rgb(color), 0.35) for L in letters]

        bars = ax.bar(x, rates, color=bar_colors, edgecolor='white', linewidth=0.5)
        ax.set_xticks(x)
        xlabels = []
        for L in letters:
            mark = '✓' if L in answer_set else ''
            xlabels.append(f'{L}{mark}\n{wrap(opts[L])}')
        ax.set_xticklabels(xlabels, fontsize=5.5)
        ax.set_ylim(0, 110)
        ax.tick_params(axis='y', labelsize=6)

        for bar, rate in zip(bars, rates):
            if rate > 0:
                ax.text(bar.get_x() + bar.get_width()/2, rate + 1, f'{rate:.0f}',
                        ha='center', fontsize=6)

        if row == 0:
            ax.set_title(f'{COND_LABELS[cond]}\n(N={n})', fontsize=8, color=color, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f'{qid}\n{tag}', fontsize=7)

plt.suptitle('Answer distributions by question and condition  |  ✓ = correct option  |  bars: full=correct, faded=incorrect',
             fontsize=10, y=1.005)
plt.tight_layout()
fig.savefig('answer_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved answer_distributions.png')

In [ ]:
# ── Correctness grids: one consolidated PNG ───────────────────────────────────
# 1 row × 5 cols, one panel per condition.

from matplotlib.colors import ListedColormap
cmap2 = ListedColormap(['#eeeeee', '#2ca02c'])

max_n = max(
    len([rec for rec in respondent_data if rec['cond'] == c])
    for c in CONDITIONS
)
fig, axes = plt.subplots(1, 5, figsize=(22, max(4, 0.42 * max_n)))

for ax, cond in zip(axes, CONDITIONS):
    recs = sorted(
        [rec for rec in respondent_data if rec['cond'] == cond],
        key=lambda r: -r['total_aon']
    )
    n = len(recs)
    if n == 0:
        ax.axis('off')
        continue

    M        = np.array([[rec[qid]['aon'] for qid in QIDS] for rec in recs])
    row_accs = M[:, [QIDS.index(q) for q in PRIMARY]].mean(axis=1)
    col_accs = M.mean(axis=0)
    pids     = [r['pid'] or f'R{i}' for i, r in enumerate(recs)]

    # Pad shorter grids to max_n so all panels have same height
    if n < max_n:
        pad = np.full((max_n - n, len(QIDS)), -1)  # -1 = empty
        M_padded = np.vstack([M, pad])
    else:
        M_padded = M

    cmap_pad = ListedColormap(['#ffffff', '#eeeeee', '#2ca02c'])
    ax.imshow(M_padded, aspect='auto', cmap=cmap_pad, vmin=-1, vmax=1,
              interpolation='nearest')

    xlabels = [f'{qid}{"*" if qid in PRIMARY else ""}\n({a:.0%})' for qid, a in zip(QIDS, col_accs)]
    ax.set_xticks(range(len(QIDS)))
    ax.set_xticklabels(xlabels, fontsize=6.5, rotation=0)
    ax.set_yticks(range(n))
    ax.set_yticklabels([f'{p}  {a:.0%}' for p, a in zip(pids, row_accs)], fontsize=6)
    if n < max_n:
        # Hide tick labels for padding rows
        ax.set_yticks(range(max_n))
        ax.set_yticklabels(
            [f'{p}  {a:.0%}' for p, a in zip(pids, row_accs)] + [''] * (max_n - n),
            fontsize=6
        )
    ax.set_title(f'{COND_LABELS[cond]}\n(N={n})', fontsize=9,
                 color=COND_COLORS[cond], fontweight='bold')

    ax.set_xticks(np.arange(-0.5, len(QIDS), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, max_n, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1)
    ax.tick_params(which='minor', length=0)

plt.suptitle('Correctness grids by condition  |  green=correct (all-or-nothing)  |  *=primary subset',
             fontsize=10, y=1.01)
plt.tight_layout()
fig.savefig('correctness_grids.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved correctness_grids.png')

In [ ]:
# ── Score comparison histograms ───────────────────────────────────────────────
# Left:  all-or-nothing count correct (out of len(PRIMARY) = 9 questions)
# Right: mean per-statement Hamming accuracy (0–1)

n_primary = len(PRIMARY)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1 — all-or-nothing
ax = axes[0]
bins_aon = np.arange(-0.5, n_primary + 1.5, 1)
for cond in CONDITIONS:
    scores = [rec['total_aon'] for rec in respondent_data if rec['cond'] == cond]
    ax.hist(scores, bins=bins_aon, alpha=0.55, label=COND_LABELS[cond],
            color=COND_COLORS[cond], density=True)
ax.set_xlabel(f'Questions correct all-or-nothing (of {n_primary} primary)')
ax.set_ylabel('Density')
ax.set_title('All-or-nothing score by condition')
ax.legend(fontsize=8)

# Add per-condition mean lines
for cond in CONDITIONS:
    scores = [rec['total_aon'] for rec in respondent_data if rec['cond'] == cond]
    if scores:
        ax.axvline(np.mean(scores), color=COND_COLORS[cond], linestyle='--', linewidth=1.5)

# Panel 2 — Hamming
ax = axes[1]
bins_ham = np.linspace(0.5, 1.01, 12)
for cond in CONDITIONS:
    scores = [rec['total_hamming'] for rec in respondent_data if rec['cond'] == cond]
    ax.hist(scores, bins=bins_ham, alpha=0.55, label=COND_LABELS[cond],
            color=COND_COLORS[cond], density=True)
ax.set_xlabel('Mean per-statement accuracy (primary subset)')
ax.set_ylabel('Density')
ax.set_title('Per-statement (Hamming) accuracy by condition')
ax.legend(fontsize=8)

for cond in CONDITIONS:
    scores = [rec['total_hamming'] for rec in respondent_data if rec['cond'] == cond]
    if scores:
        ax.axvline(np.mean(scores), color=COND_COLORS[cond], linestyle='--', linewidth=1.5)

plt.tight_layout()
fig.savefig('score_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print(f'{"Condition":<22} {"N":>4}  {"Mean AoN":>9}  {"Mean Ham":>9}')
print('-' * 50)
for cond in CONDITIONS:
    recs = [rec for rec in respondent_data if rec['cond'] == cond]
    n    = len(recs)
    if not recs:
        continue
    mean_aon = np.mean([r['total_aon']    for r in recs])
    mean_ham = np.mean([r['total_hamming'] for r in recs])
    print(f'{COND_LABELS[cond]:<22} {n:>4}  {mean_aon:>9.2f}  {mean_ham:>9.3f}')

print(f'\nPrimary subset ({len(PRIMARY)} questions): {PRIMARY}')
print('AoN = all-or-nothing count correct; Ham = mean per-statement accuracy')